# Lab 2 -  Feature Selection


In [1]:
import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [9]:
# ── Data Loading: F1 Results + Qualifying via Ergast API ─────────────────────
import requests
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
SEASONS   = [2022, 2023, 2024]
FEATURES  = ['grid', 'driver', 'q3_time_sec']
TARGET    = 'top10'
BASE_URL  = "https://api.jolpi.ca/ergast/f1"

# ── Helpers ───────────────────────────────────────────────────────────────────
def _paginate(url_template: str, table_key: str, row_key: str) -> list:
    """Generic paginator for any Ergast endpoint."""
    offset, limit, rows = 0, 100, []
    while True:
        url  = f"{url_template}?limit={limit}&offset={offset}"
        data = requests.get(url, timeout=30).raise_for_status() or \
               requests.get(url, timeout=30).json()['MRData']
        # cleaner version:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data  = resp.json()['MRData']
        total = int(data['total'])
        rows.extend(data[table_key][row_key])
        offset += limit
        if offset >= total:
            break
    return rows

def time_to_sec(t: str) -> float | None:
    """Convert 'm:ss.mmm' string to total seconds."""
    if not t:
        return None
    try:
        m, s = t.split(':')
        return int(m) * 60 + float(s)
    except Exception:
        return None

# ── Loaders ───────────────────────────────────────────────────────────────────
def get_season_results(year: int) -> pd.DataFrame:
    rows = []
    for race in _paginate(f"{BASE_URL}/{year}/results.json", 'RaceTable', 'Races'):
        for r in race['Results']:
            rows.append({
                'season'    : int(race['season']),
                'round'     : int(race['round']),
                'race_name' : race['raceName'],
                'circuit'   : race['Circuit']['circuitId'],
                'date'      : race['date'],
                'driver'    : r['Driver']['driverId'],
                'driver_name': f"{r['Driver']['givenName']} {r['Driver']['familyName']}",
                'constructor': r['Constructor']['constructorId'],
                'grid'      : int(r['grid']),
                'position'  : int(r['position']) if r['position'].isdigit() else None,
                'points'    : float(r['points']),
                'status'    : r['status'],
                'laps'      : int(r['laps']),
            })
    return pd.DataFrame(rows)


def get_qualifying_results(year: int) -> pd.DataFrame:
    rows = []
    for race in _paginate(f"{BASE_URL}/{year}/qualifying.json", 'RaceTable', 'Races'):
        for r in race['QualifyingResults']:
            rows.append({
                'season'     : int(race['season']),
                'round'      : int(race['round']),
                'driver'     : r['Driver']['driverId'],
                'quali_pos'  : int(r['position']),
                'q1_time_sec': time_to_sec(r.get('Q1')),
                'q2_time_sec': time_to_sec(r.get('Q2')),
                'q3_time_sec': time_to_sec(r.get('Q3')),   # None si eliminado antes
            })
    return pd.DataFrame(rows)

# ── Load & merge ──────────────────────────────────────────────────────────────
print("Loading F1 data...")

results, qualis = [], []
for year in SEASONS:
    df_r = get_season_results(year)
    df_q = get_qualifying_results(year)
    print(f"  {year}: {len(df_r)} results | {len(df_q)} quali entries")
    results.append(df_r)
    qualis.append(df_q)

df_results = pd.concat(results, ignore_index=True)
df_quali   = pd.concat(qualis,  ignore_index=True)

df_all = df_results.merge(
    df_quali[['season', 'round', 'driver',
              'quali_pos', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec']],
    on=['season', 'round', 'driver'],
    how='left'
)

# ── Feature engineering ───────────────────────────────────────────────────────
df_all['date']  = pd.to_datetime(df_all['date'])
df_all[TARGET]  = (df_all['position'] <= 10)

# Encode driver como categoría ordinal (modelo puede usarlo directo)
df_all['driver_enc'] = df_all['driver'].astype('category').cat.codes

# ── Dataset listo para modelo ─────────────────────────────────────────────────
# Nota: q3_time_sec es NaN para pilotos eliminados en Q1/Q2 → tratar antes de modelar
FEATURE_COLS = {
    'grid'       : 'grid',          # posición de largada
    'driver'     : 'driver_enc',    # piloto codificado
    'q3_time_sec': 'q3_time_sec',   # tiempo Q3 en segundos (NaN si no clasificó Q3)
}

X = df_all[list(FEATURE_COLS.values())]
y = df_all[TARGET]

print(f"\nDataset final: {len(df_all)} filas | {X.isna().sum().to_dict()} NaNs por feature")
print(df_all[['race_name', 'driver', 'grid', 'q3_time_sec', 'driver_enc', TARGET]].head(10))

Loading F1 data...
  2022: 440 results | 440 quali entries
  2023: 440 results | 440 quali entries
  2024: 479 results | 479 quali entries

Dataset final: 1359 filas | {'grid': 0, 'driver_enc': 0, 'q3_time_sec': 698} NaNs por feature
            race_name           driver  grid  q3_time_sec  driver_enc  top10
0  Bahrain Grand Prix          leclerc     1       90.558          13   True
1  Bahrain Grand Prix            sainz     3       90.687          22   True
2  Bahrain Grand Prix         hamilton     5       91.238           8   True
3  Bahrain Grand Prix          russell     9       92.216          21   True
4  Bahrain Grand Prix  kevin_magnussen     7       91.808          10   True
5  Bahrain Grand Prix           bottas     6       91.560           3   True
6  Bahrain Grand Prix             ocon    11          NaN          17   True
7  Bahrain Grand Prix          tsunoda    16          NaN          25   True
8  Bahrain Grand Prix           alonso     8       92.195           1   T

In [8]:
df_all.columns

Index(['season', 'round', 'race_name', 'circuit', 'date', 'driver',
       'driver_name', 'constructor', 'grid', 'position', 'points', 'status',
       'laps', 'top10'],
      dtype='object')

## Feature selection

We choose the features

**grid**:Because the initial positión show be important that is better than toss a coin, so its the dfirst parámeter

**driver**:The driver across the time its useful for select the top drivers that you can expect almost always to win

**q3_time**:The last time that the drivers can prove themselfs its key to predict they result.

For the target top10

In [12]:
# ── Logistic Regression: Top 10 Predictor ────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_score, recall_score



import warnings
warnings.filterwarnings('ignore')

# ── Preprocessing ─────────────────────────────────────────────────────────────
def prepare_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """
    Prepara features desde el DataFrame mergeado de results + qualifying.
    Espera columnas: grid, driver, q3_time_sec, top10
    """
    df = df.copy()

    # 1. Encode driver (LabelEncoder reproducible)
    le = LabelEncoder()
    df['driver_enc'] = le.fit_transform(df['driver'])

    # 2. Imputar q3_time_sec: peor tiempo Q3 de esa carrera + flag binario
    df['reached_q3'] = df['q3_time_sec'].notna().astype(int)
    df['q3_time_sec'] = df.groupby(['season', 'round'])['q3_time_sec'] \
                          .transform(lambda x: x.fillna(x.max()))

    # 3. Seleccionar features finales
    feature_cols = ['grid', 'driver_enc', 'q3_time_sec']
    X = df[feature_cols]
    y = df['top10']

    return X, y, le


# ── Model ─────────────────────────────────────────────────────────────────────
def build_pipeline() -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),          # normaliza grid y q3_time_sec
        ('model',  LogisticRegression(
            class_weight='balanced',           # corrige desbalance ~10 top10 vs ~10 no-top10
            max_iter=1000,
            random_state=42,
        ))
    ])


# ── Train & Evaluate ──────────────────────────────────────────────────────────
def train_and_evaluate(df: pd.DataFrame) -> dict:
    X, y, le = prepare_features(df)

    print(f"Dataset: {len(df)} filas | Target balance: {y.mean():.1%} top10\n")
 
    y_pred = results['pipeline'].predict(X)
    precision = precision_score(y, y_pred)

    pipeline = build_pipeline()

    # Cross-validation estratificada (mantiene balance de clases por fold)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_auc    = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')
    cv_f1     = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')
    cv_acc    = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

    print("── Cross-Validation (5-fold) ──────────────────────────")
    print(f"  ROC-AUC : {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
    print(f"  F1      : {cv_f1.mean():.3f}  ± {cv_f1.std():.3f}")
    print(f"  Accuracy: {cv_acc.mean():.3f} ± {cv_acc.std():.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall_score(y, y_pred):.3f}")

    # Fit final sobre todo el dataset
    pipeline.fit(X, y)

    # Coeficientes interpretables
    coefs = pd.Series(
        pipeline.named_steps['model'].coef_[0],
        index=['grid', 'driver_enc', 'q3_time_sec']
    ).sort_values(key=abs, ascending=False)

    print("\n── Coeficientes (mayor |valor| = más influyente) ──────")
    for feat, coef in coefs.items():
        direction = "↑ top10" if coef < 0 else "↓ top10"
        print(f"  {feat:<15} {coef:+.4f}  ({direction} al aumentar)")

    # Reporte completo sobre todo el set
    y_pred  = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)[:, 1]

    print("\n── Classification Report (full dataset) ───────────────")
    print(classification_report(y, y_pred, target_names=['No Top10', 'Top10']))
    print(f"ROC-AUC (full): {roc_auc_score(y, y_proba):.3f}")

    return {
        'pipeline' : pipeline,
        'encoder'  : le,
        'cv_auc'   : cv_auc,
        'coefs'    : coefs,
    }


# ── Predict new entries ───────────────────────────────────────────────────────
def predict_race(pipeline, le: LabelEncoder,
                 entries: list[dict]) -> pd.DataFrame:
    """
    Predice probabilidad top10 para una lista de entradas.

    entries ejemplo:
    [
        {'driver': 'max_verstappen', 'grid': 1, 'q3_time_sec': 88.2},
        {'driver': 'leclerc',        'grid': 3, 'q3_time_sec': 88.5},
        {'driver': 'alonso',         'grid': 15, 'q3_time_sec': None},  # no llegó a Q3
    ]
    """
    df_pred = pd.DataFrame(entries)

    # Encode driver (unseen drivers → -1)
    df_pred['driver_enc'] = df_pred['driver'].apply(
        lambda d: le.transform([d])[0] if d in le.classes_ else -1
    )

    # Imputar q3_time_sec faltante con el peor tiempo del grupo
    if df_pred['q3_time_sec'].isna().any():
        worst = df_pred['q3_time_sec'].max()
        df_pred['q3_time_sec'] = df_pred['q3_time_sec'].fillna(worst)

    X_pred = df_pred[['grid', 'driver_enc', 'q3_time_sec']]
    df_pred['p_top10'] = pipeline.predict_proba(X_pred)[:, 1]
    df_pred['pred_top10'] = pipeline.predict(X_pred)

    return df_pred[['driver', 'grid', 'q3_time_sec', 'p_top10', 'pred_top10']] \
               .sort_values('p_top10', ascending=False) \
               .reset_index(drop=True)


# ── Run ───────────────────────────────────────────────────────────────────────
results = train_and_evaluate(df_all)

# Ejemplo de predicción para una carrera nueva
sample_entries = [
    {'driver': 'max_verstappen', 'grid': 1,  'q3_time_sec': 88.2},
    {'driver': 'leclerc',        'grid': 2,  'q3_time_sec': 88.4},
    {'driver': 'hamilton',       'grid': 5,  'q3_time_sec': 88.7},
    {'driver': 'alonso',         'grid': 12, 'q3_time_sec': None},
]

print("\n── Predicción carrera nueva ────────────────────────────")
print(predict_race(results['pipeline'], results['encoder'], sample_entries))

Dataset: 1359 filas | Target balance: 50.0% top10

── Cross-Validation (5-fold) ──────────────────────────
  ROC-AUC : 0.813 ± 0.035
  F1      : 0.762  ± 0.032
  Accuracy: 0.762 ± 0.029
Precision: 0.759
Recall   : 0.765

── Coeficientes (mayor |valor| = más influyente) ──────
  grid            -1.3499  (↑ top10 al aumentar)
  driver_enc      +0.0423  (↓ top10 al aumentar)
  q3_time_sec     -0.0308  (↑ top10 al aumentar)

── Classification Report (full dataset) ───────────────
              precision    recall  f1-score   support

    No Top10       0.76      0.76      0.76       679
       Top10       0.76      0.76      0.76       680

    accuracy                           0.76      1359
   macro avg       0.76      0.76      0.76      1359
weighted avg       0.76      0.76      0.76      1359

ROC-AUC (full): 0.815

── Predicción carrera nueva ────────────────────────────
           driver  grid  q3_time_sec   p_top10  pred_top10
0  max_verstappen     1         88.2  0.896352       